In [42]:
import numpy as np
from scipy.special import betaln, expit, logsumexp
from scipy.optimize import minimize
from sklearn.model_selection import KFold

# 1. Realistic AIME-style Dataset (30 problems, 10 attempts)
# 15 zeros, 10 partials, 5 perfects
successes = np.array([0]*15 + [1, 1, 2, 2, 3, 4, 5, 7, 8, 9] + [10]*5)
attempts = np.array([10]*30)

def fit_1_component(y, n):
    """Fits a single Beta-Binomial."""
    def nll(log_params):
        a, b = np.exp(log_params)
        ll = betaln(y + a, n - y + b) - betaln(a, b)
        return -np.sum(ll)
    
    # Init with moments roughly spanning the space
    res = minimize(nll, [np.log(0.5), np.log(1.0)], method='L-BFGS-B')
    return np.exp(res.x)

def fit_2_component(y, n):
    """Fits a 2-Component Beta-Binomial Mixture."""
    def nll(params):
        w1 = expit(params[0])
        w2 = 1.0 - w1
        a1, b1, a2, b2 = np.exp(params[1:])
        
        ll1 = betaln(y + a1, n - y + b1) - betaln(a1, b1)
        ll2 = betaln(y + a2, n - y + b2) - betaln(a2, b2)
        
        # log(w1*p1 + w2*p2) = logsumexp([log(w1)+ll1, log(w2)+ll2])
        mix_ll = logsumexp(np.vstack([np.log(w1) + ll1, np.log(w2) + ll2]), axis=0)
        return -np.sum(mix_ll)
    
    # Smart Init: Split data into low/high to encourage separation
    p_hat = y / n
    low_p = p_hat[p_hat < 0.5]; high_p = p_hat[p_hat >= 0.5]
    
    a1_init = np.mean(low_p) if len(low_p)>0 else 0.1
    b1_init = 1 - a1_init
    a2_init = np.mean(high_p) if len(high_p)>0 else 0.9
    b2_init = 1 - a2_init

    init_params = [
        0.0, # logit(w) = 0.5
        np.log(a1_init+0.1), np.log(b1_init+0.1),
        np.log(a2_init+0.1), np.log(b2_init+0.1)
    ]
    
    res = minimize(nll, init_params, method='L-BFGS-B')
    w1 = expit(res.x[0])
    params = np.exp(res.x[1:])
    return w1, params

def extrapolate_pass_at_k(a, b, k=1000):
    """Calculates Pass@k for a given Beta component."""
    log_fail = betaln(a, b + k) - betaln(a, b)
    return 1.0 - np.exp(log_fail)

# --- EXECUTE EVALUATION ---

print("=== Training Metrics (All 30 Prompts) ===")
# 1-Comp Fit
a_single, b_single = fit_1_component(successes, attempts)
train_nll_single = -np.sum(betaln(successes + a_single, attempts - successes + b_single) - betaln(a_single, b_single))

# 2-Comp Fit
w1, params_2c = fit_2_component(successes, attempts)
a1, b1, a2, b2 = params_2c
ll1 = betaln(successes + a1, attempts - successes + b1) - betaln(a1, b1)
ll2 = betaln(successes + a2, attempts - successes + b2) - betaln(a2, b2)
train_nll_mix = -np.sum(logsumexp(np.vstack([np.log(w1) + ll1, np.log(1-w1) + ll2]), axis=0))

print(f"1-Component Train NLL: {train_nll_single:.4f}")
print(f"2-Component Train NLL: {train_nll_mix:.4f}")
print(f"2-Component Params: w1={w1:.2f}, Comp1(a={a1:.3f}, b={b1:.3f}), Comp2(a={a2:.3f}, b={b2:.3f})")

print("\n=== 5-Fold Cross-Validation NLL ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_nll_single = []
cv_nll_mix = []

for train_idx, val_idx in kf.split(successes):
    y_tr, n_tr = successes[train_idx], attempts[train_idx]
    y_val, n_val = successes[val_idx], attempts[val_idx]
    
    # Single Fit & Eval
    a_s, b_s = fit_1_component(y_tr, n_tr)
    val_ll_s = betaln(y_val + a_s, n_val - y_val + b_s) - betaln(a_s, b_s)
    cv_nll_single.append(-np.sum(val_ll_s))
    
    # Mix Fit & Eval
    w_m, p_m = fit_2_component(y_tr, n_tr)
    a1m, b1m, a2m, b2m = p_m
    val_ll1 = betaln(y_val + a1m, n_val - y_val + b1m) - betaln(a1m, b1m)
    val_ll2 = betaln(y_val + a2m, n_val - y_val + b2m) - betaln(a2m, b2m)
    val_ll_m = logsumexp(np.vstack([np.log(w_m) + val_ll1, np.log(1-w_m) + val_ll2]), axis=0)
    cv_nll_mix.append(-np.sum(val_ll_m))

print(f"1-Component Mean Val NLL: {np.mean(cv_nll_single):.4f}")
print(f"2-Component Mean Val NLL: {np.mean(cv_nll_mix):.4f}")

print("\n=== Extrapolation Test (Pass@1000) ===")
# Evaluate extrapolation for the cluster managing the 0/10 prompts
pass1k_single = extrapolate_pass_at_k(a_single, b_single, 1000)
pass1k_comp1 = extrapolate_pass_at_k(a1, b1, 1000)  # The component assigned to the left tail

print(f"1-Component Predicted Pass@1000: {pass1k_single:.2%}")
print(f"2-Component Predicted Pass@1000: {pass1k_comp1:.2%} (from its left-most component)")

=== Training Metrics (All 30 Prompts) ===
1-Component Train NLL: 90.4864
2-Component Train NLL: 90.3460
2-Component Params: w1=0.66, Comp1(a=0.144, b=1.388), Comp2(a=0.489, b=0.184)

=== 5-Fold Cross-Validation NLL ===
1-Component Mean Val NLL: 18.6936
2-Component Mean Val NLL: 19.5269

=== Extrapolation Test (Pass@1000) ===
1-Component Predicted Pass@1000: 72.05%
2-Component Predicted Pass@1000: 63.13% (from its left-most component)


/tmp/ipykernel_181790/1299821423.py:30: RuntimeWarning: invalid value encountered in subtract
  ll2 = betaln(y + a2, n - y + b2) - betaln(a2, b2)
/tmp/ipykernel_181790/1299821423.py:27: RuntimeWarning: overflow encountered in exp
  a1, b1, a2, b2 = np.exp(params[1:])
/tmp/ipykernel_181790/1299821423.py:33: RuntimeWarning: divide by zero encountered in log
  mix_ll = logsumexp(np.vstack([np.log(w1) + ll1, np.log(w2) + ll2]), axis=0)


In [ ]:
import numpy as np
from scipy.special import betaln, expit, logsumexp
from scipy.optimize import minimize
from sklearn.model_selection import KFold

# === 1. SIMULATE GROUND TRUTH ===
# np.random.seed(44)  # Fixed seed so you see the exact collapse
N_prompts = 1000
m_attempts = 10

# The True Latent Physics (A 2-Component Beta Mixture)
true_w1, true_w2 = 0.70, 0.30
true_a1, true_b1 = 0.3, 3.0  # Hard prompts: massive density at 0, heavy tail
true_a2, true_b2 = 5.0, 1.0  # Easy prompts: smooth peak near 0.8

# Generate 30 prompts from the true latent distributions
clusters = np.random.binomial(1, true_w2, size=N_prompts)
true_thetas = np.where(
    clusters == 0,
    np.random.beta(true_a1, true_b1, size=N_prompts),
    np.random.beta(true_a2, true_b2, size=N_prompts)
)

# Generate observable counts (m=10)
successes = np.random.binomial(m_attempts, true_thetas)
attempts = np.full(N_prompts, m_attempts)

print(f"Empirical Success Counts:\n{np.sort(successes)}\n")

# === 2. THE MATHEMATICAL FITS ===
# We use np.clip(..., 1e-5, 10000) to safely catch the Dirac Singularity
def fit_1_comp(y, n):
    def nll(p):
        a, b = np.clip(np.exp(p), 1e-5, 10000.0)
        return -np.sum(betaln(y+a, n-y+b) - betaln(a,b))
    res = minimize(nll, [0.0, 0.0], method='L-BFGS-B')
    return np.clip(np.exp(res.x), 1e-5, 10000.0)

def fit_2_comp(y, n):
    def nll(p):
        w1 = expit(p[0])
        w2 = 1.0 - w1
        a1, b1, a2, b2 = np.clip(np.exp(p[1:]), 1e-5, 10000.0)
        ll1 = betaln(y+a1, n-y+b1) - betaln(a1,b1)
        ll2 = betaln(y+a2, n-y+b2) - betaln(a2,b2)
        return -np.sum(logsumexp([np.log(w1)+ll1, np.log(w2)+ll2], axis=0))
    
    # Initialize smoothly
    init_p = [0.0, np.log(0.1), np.log(0.9), np.log(0.8), np.log(0.2)]
    res = minimize(nll, init_p, method='L-BFGS-B')
    w1 = expit(res.x[0])
    return w1, np.clip(np.exp(res.x[1:]), 1e-5, 10000.0)

# === 3. PASS@100 EXTRAPOLATION CALCULATION ===
def pass_at_100_single(a, b):
    return 1.0 - np.exp(betaln(a, b + 20) - betaln(a, b))

def pass_at_100_mix(w1, a1, b1, a2, b2):
    fail1 = np.exp(betaln(a1, b1 + 20) - betaln(a1, b1))
    fail2 = np.exp(betaln(a2, b2 + 20) - betaln(a2, b2))
    return 1.0 - (w1 * fail1 + (1.0 - w1) * fail2)

# Calculate True Pass@100
true_pass_100 = pass_at_100_mix(true_w1, true_a1, true_b1, true_a2, true_b2)

# Fit Models
a_s, b_s = fit_1_comp(successes, attempts)
w1_m, p_m = fit_2_comp(successes, attempts)
a1_m, b1_m, a2_m, b2_m = p_m

# Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_single, cv_mix = [], []
for tr, val in kf.split(successes):
    y_tr, n_tr, y_val, n_val = successes[tr], attempts[tr], successes[val], attempts[val]
    # Single
    a_tr, b_tr = fit_1_comp(y_tr, n_tr)
    cv_single.append(-np.sum(betaln(y_val+a_tr, n_val-y_val+b_tr) - betaln(a_tr, b_tr)))
    # Mix
    w_tr, p_tr = fit_2_comp(y_tr, n_tr)
    a1_t, b1_t, a2_t, b2_t = p_tr
    ll1 = betaln(y_val+a1_t, n_val-y_val+b1_t) - betaln(a1_t,b1_t)
    ll2 = betaln(y_val+a2_t, n_val-y_val+b2_t) - betaln(a2_t,b2_t)
    cv_mix.append(-np.sum(logsumexp([np.log(w_tr)+ll1, np.log(1-w_tr)+ll2], axis=0)))

print("=== The Ground Truth ===")
print(f"True Pass@100: {true_pass_100:.2%}")
print(f"True Params: w1={true_w1:.2f}, Comp1(a={true_a1:.1f}, b={true_b1:.1f}), Comp2(a={true_a2:.1f}, b={true_b2:.1f})\n")

print("=== The Recovered Params ===")
print(f"1-Comp Fit: a={a_s:.3f}, b={b_s:.3f}")
print(f"2-Comp Fit: w1={w1_m:.2f}, Comp1(a={a1_m:.3f}, b={b1_m:.3f}), Comp2(a={a2_m:.3f}, b={b2_m:.3f})\n")

print("=== 5-Fold Cross Validation NLL ===")
print(f"1-Comp Mean Val NLL: {np.mean(cv_single):.4f}")
print(f"2-Comp Mean Val NLL: {np.mean(cv_mix):.4f}\n")

print("=== Predicted Extrapolation ===")
print(f"1-Comp Pass@100: {pass_at_100_single(a_s, b_s):.2%}")
print(f"2-Comp Pass@100: {pass_at_100_mix(w1_m, a1_m, b1_m, a2_m, b2_m):.2%}")

Empirical Success Counts:
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0 